In [1]:
# ============================================
# IPL Win Probability Predictor
# Day 3 — Data Cleaning
# Phase 4 — Cleaning Matches + Building Full Dataset
# ============================================
# GOAL: Remove messy matches, merge matches.csv with 
# deliveries.csv, and apply Phase 3 logic to ALL matches

import pandas as pd
import numpy as np

matches = pd.read_csv("../data/matches.csv")
deliveries = pd.read_csv("../data/deliveries.csv")

print("✅ Datasets reloaded")
print("Matches:", matches.shape)
print("Deliveries:", deliveries.shape)

✅ Datasets reloaded
Matches: (756, 18)
Deliveries: (179078, 21)


In [2]:
# Fix inconsistent team names in deliveries data
team_name_fixes = {
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Delhi Capitals': 'Delhi Daredevils',  # check if this exists too
}

deliveries['batting_team'] = deliveries['batting_team'].replace(team_name_fixes)
deliveries['bowling_team'] = deliveries['bowling_team'].replace(team_name_fixes)

print("Team name fixes applied:")
for old, new in team_name_fixes.items():
    print(f"  '{old}' → '{new}'")

print("\nUnique teams after fix:")
print(sorted(deliveries['batting_team'].unique()))

Team name fixes applied:
  'Rising Pune Supergiant' → 'Rising Pune Supergiants'
  'Delhi Capitals' → 'Delhi Daredevils'

Unique teams after fix:
['Chennai Super Kings', 'Deccan Chargers', 'Delhi Daredevils', 'Gujarat Lions', 'Kings XI Punjab', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Mumbai Indians', 'Pune Warriors', 'Rajasthan Royals', 'Rising Pune Supergiants', 'Royal Challengers Bangalore', 'Sunrisers Hyderabad']


In [3]:
# Clean matches dataset - keep only normal results, no DL, valid winner
matches_clean = matches[matches['result'] == 'normal']
matches_clean = matches_clean[matches_clean['dl_applied'] == 0]
matches_clean = matches_clean[matches_clean['winner'].notna()]

print("Original matches:", matches.shape[0])
print("After cleaning:", matches_clean.shape[0])
print("Removed:", matches.shape[0] - matches_clean.shape[0])

Original matches: 756
After cleaning: 724
Removed: 32


In [4]:
# We only care about chase situations (innings 2)
deliveries_inning2=deliveries[deliveries['inning']==2]

print("Total deliveries (all inning) :",deliveries.shape[0])
print("Inning 2 only :",deliveries_inning2.shape[0])

Total deliveries (all inning) : 179078
Inning 2 only : 86240


In [5]:
# Calculate innings 1 total score for EVERY match
innings1_total=deliveries[deliveries['inning']==1].groupby('match_id')['total_runs'].sum()

# Convert to a clean DataFrame: match_id -> target
innings1_total=innings1_total.reset_index()
innings1_total.columns=['match_id','inning_score']
innings1_total['target']=innings1_total['inning_score']+1

innings1_total.head()

,match_id,inning_score,target
0,1,207,208
1,2,184,185
2,3,183,184
3,4,163,164
4,5,157,158


In [6]:
# Merge target info into our innings 2 deliveries
deliveries_inning2=deliveries_inning2.merge(innings1_total[['match_id','target']],on='match_id',how='left')

deliveries_inning2[['match_id', 'inning', 'over', 'ball', 'total_runs', 'target']].head()

,match_id,inning,over,ball,total_runs,target
0,1,2,1,1,1,208
1,1,2,1,2,0,208
2,1,2,1,3,0,208
3,1,2,1,4,2,208
4,1,2,1,5,4,208


In [7]:
# Bring in winner and teams info from matches_clean
deliveries_inning2=deliveries_inning2.merge(matches_clean[['id','winner','team1','team2']],left_on='match_id',right_on='id',how='inner')

deliveries_inning2[['match_id', 'batting_team', 'bowling_team', 'winner', 'target']].head()

,match_id,batting_team,bowling_team,winner,target
0,1,Royal Challengers Bangalore,Sunrisers Hyderabad,Sunrisers Hyderabad,208
1,1,Royal Challengers Bangalore,Sunrisers Hyderabad,Sunrisers Hyderabad,208
2,1,Royal Challengers Bangalore,Sunrisers Hyderabad,Sunrisers Hyderabad,208
3,1,Royal Challengers Bangalore,Sunrisers Hyderabad,Sunrisers Hyderabad,208
4,1,Royal Challengers Bangalore,Sunrisers Hyderabad,Sunrisers Hyderabad,208


In [8]:
print("Shape after merging:", deliveries_inning2.shape)
print("\nUnique matches remaining:", deliveries_inning2['match_id'].nunique())
print("\nAny missing targets?", deliveries_inning2['target'].isna().sum())
print("Any missing winners?", deliveries_inning2['winner'].isna().sum())

Shape after merging: (83933, 26)

Unique matches remaining: 724

Any missing targets? 0
Any missing winners? 0


In [9]:
# Save the cleaned, merged dataset for future use
deliveries_inning2.to_csv("../data/cleaned_inning2.csv", index=False)
print("✅ Saved cleaned dataset")
print("Shape saved:", deliveries_inning2.shape)

✅ Saved cleaned dataset
Shape saved: (83933, 26)
